# WC 2026 Predictor -- Exploratory Data Analysis

Exploring the historical match dataset before feature engineering: distribution of outcomes,
goal-scoring trends, and how rank differential relates to match results.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

BASE_DIR = os.path.dirname(os.getcwd())
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')

df = pd.read_csv(os.path.join(PROCESSED_DIR, 'matches_features_final.csv'))
df['MatchDate'] = pd.to_datetime(df['MatchDate'])
df.head()

## 1. Outcome distribution
How often do home teams win vs draw vs lose, across 47K+ international matches?

In [ ]:
result_labels = {0: 'Home Win', 1: 'Draw', 2: 'Away Win'}
df['ResultLabel'] = df['Result'].map(result_labels)

plt.figure(figsize=(6,4))
sns.countplot(data=df, x='ResultLabel', order=['Home Win', 'Draw', 'Away Win'])
plt.title('Match Outcome Distribution (1996-2024)')
plt.ylabel('Number of Matches')
plt.xlabel('')
plt.show()

df['ResultLabel'].value_counts(normalize=True) * 100

## 2. Does rank differential actually predict outcomes?
This is the core hypothesis the model is built on -- worth checking visually before trusting it.

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='ResultLabel', y='RankDiff', order=['Home Win', 'Draw', 'Away Win'])
plt.axhline(0, color='gray', linestyle='--')
plt.title('Rank Differential by Match Outcome')
plt.ylabel('Rank Diff (positive = home team ranked better)')
plt.xlabel('')
plt.show()

**Reading this chart:** if the hypothesis holds, Home Win should skew toward positive RankDiff, Away Win toward negative, and Draw should sit closer to zero (evenly matched teams). This is what justifies using RankDiff as the top model feature.

## 3. Home advantage over time
Is home advantage still meaningful in modern international football, or has it eroded?

In [ ]:
df['Year'] = df['MatchDate'].dt.year
yearly_home_win_rate = df[df['IsNeutralVenue'] == 0].groupby('Year')['Result'].apply(lambda x: (x == 0).mean())

plt.figure(figsize=(10,5))
yearly_home_win_rate.plot()
plt.title('Home Win Rate by Year (Non-Neutral Venues Only)')
plt.ylabel('Home Win Rate')
plt.xlabel('Year')
plt.axhline(yearly_home_win_rate.mean(), color='red', linestyle='--', label='Overall Average')
plt.legend()
plt.show()

## 4. Head-to-head signal check
Does historical head-to-head win rate actually correlate with the current match result, or is it mostly noise?

In [ ]:
h2h_data = df.dropna(subset=['H2H_HomeWinPct'])

plt.figure(figsize=(8,5))
sns.boxplot(data=h2h_data, x='ResultLabel', y='H2H_HomeWinPct', order=['Home Win', 'Draw', 'Away Win'])
plt.title('Historical H2H Win % by Current Match Outcome')
plt.ylabel('Home Team Historical H2H Win Rate vs This Opponent')
plt.xlabel('')
plt.show()

## Summary

- Match outcomes are imbalanced (~45% Home Win, ~25% Draw, ~30% Away Win), consistent with well-known home advantage in football.
- RankDiff shows a clear directional relationship with outcome -- validates it as the strongest model feature (confirmed later in feature importance).
- Head-to-head history shows a visible but weaker relationship -- useful signal, not dominant on its own.

These findings directly informed the feature set used in `03_feature_engineering.py` and explain why RankDiff and PointsDiff dominate feature importance in the trained model.